In [1]:
# ========================================
# 0.Imports
# ========================================

import pandas as pd
import numpy as np


In [2]:
# ========================================
# 1.Reading the .csv files
# ========================================

sessions_data = pd.read_csv('../data/raw/sessions_data.csv')
web_data = pd.read_csv('../data/clean/cleaned_web_data.csv')
clients_data = pd.read_csv('../data/clean/cleaned_client_data.csv')

In [3]:
web_data[["client_id", "visit_id", "process_step", "date_time", "variation"]].head(10)

,client_id,visit_id,process_step,date_time,variation
0,555,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test
1,555,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test
2,555,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test
3,555,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test
4,555,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test
5,647,40369564_40101682850_311847,confirm,2017-04-12 15:47:45,Test
6,647,40369564_40101682850_311847,step_3,2017-04-12 15:45:02,Test
7,647,40369564_40101682850_311847,step_2,2017-04-12 15:41:53,Test
8,647,40369564_40101682850_311847,step_1,2017-04-12 15:41:35,Test
9,647,40369564_40101682850_311847,start,2017-04-12 15:41:28,Test


In [4]:
# ========================================
# 2.Calculate the duration and related statistics of every step
# ========================================

In [5]:
# We check the data type of date_time & correct it
web_data["date_time"].dtype

<StringDtype(storage='python', na_value=nan)>

In [6]:
web_data["date_time"] = pd.to_datetime(web_data["date_time"])
web_data["date_time"].dtype

dtype('<M8[us]')

In [7]:
# We calculate the duration of every step
# First, we group the records by client and visit with the earliest timestamp first

timed_steps = (
    web_data
    .sort_values(["client_id", "visit_id", "date_time"])
    .reset_index(drop=True)
)

timed_steps[["client_id", "visit_id", "process_step", "date_time", "variation"]].head(10)

,client_id,visit_id,process_step,date_time,variation
0,555,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test
1,555,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test
2,555,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test
3,555,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test
4,555,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test
5,647,40369564_40101682850_311847,start,2017-04-12 15:41:28,Test
6,647,40369564_40101682850_311847,step_1,2017-04-12 15:41:35,Test
7,647,40369564_40101682850_311847,step_2,2017-04-12 15:41:53,Test
8,647,40369564_40101682850_311847,step_3,2017-04-12 15:45:02,Test
9,647,40369564_40101682850_311847,confirm,2017-04-12 15:47:45,Test


In [8]:
# We add a new column with the following timestamp so we can compare them after

timed_steps["next_step_time"] = (
    timed_steps.groupby(["client_id", "visit_id"])["date_time"].shift(-1)
)

timed_steps[["process_step", "date_time", "next_step_time", "variation"]].head(10)

,process_step,date_time,next_step_time,variation
0,start,2017-04-15 12:57:56,2017-04-15 12:58:03,Test
1,step_1,2017-04-15 12:58:03,2017-04-15 12:58:35,Test
2,step_2,2017-04-15 12:58:35,2017-04-15 13:00:14,Test
3,step_3,2017-04-15 13:00:14,2017-04-15 13:00:34,Test
4,confirm,2017-04-15 13:00:34,NaT,Test
5,start,2017-04-12 15:41:28,2017-04-12 15:41:35,Test
6,step_1,2017-04-12 15:41:35,2017-04-12 15:41:53,Test
7,step_2,2017-04-12 15:41:53,2017-04-12 15:45:02,Test
8,step_3,2017-04-12 15:45:02,2017-04-12 15:47:45,Test
9,confirm,2017-04-12 15:47:45,NaT,Test


In [9]:
# We calculate the time passed between steps and add a new column to display it

timed_steps["step_duration"] = (
    timed_steps["next_step_time"] - timed_steps["date_time"]
).dt.total_seconds()

timed_steps[["process_step", "date_time", "next_step_time", "step_duration", "variation"]].head(10)

,process_step,date_time,next_step_time,step_duration,variation
0,start,2017-04-15 12:57:56,2017-04-15 12:58:03,7.0,Test
1,step_1,2017-04-15 12:58:03,2017-04-15 12:58:35,32.0,Test
2,step_2,2017-04-15 12:58:35,2017-04-15 13:00:14,99.0,Test
3,step_3,2017-04-15 13:00:14,2017-04-15 13:00:34,20.0,Test
4,confirm,2017-04-15 13:00:34,NaT,NaN,Test
5,start,2017-04-12 15:41:28,2017-04-12 15:41:35,7.0,Test
6,step_1,2017-04-12 15:41:35,2017-04-12 15:41:53,18.0,Test
7,step_2,2017-04-12 15:41:53,2017-04-12 15:45:02,189.0,Test
8,step_3,2017-04-12 15:45:02,2017-04-12 15:47:45,163.0,Test
9,confirm,2017-04-12 15:47:45,NaT,NaN,Test


In [10]:
# We want to check how many sessions reached a specific step in the user journey
sessions_by_step = (timed_steps.drop_duplicates()
    .pivot_table(
        index="process_step",
        columns="variation",
        aggfunc="size"
    )
    .reindex(["start", "step_1", "step_2", "step_3", "confirm"])
)

display(sessions_by_step)

variation,Control,Test
process_step,,
start,45380,55773
step_1,29544,38666
step_2,25773,30899
step_3,22503,25761
confirm,17336,25600


In [11]:
# Check the median time for each step for users in group Test vs Control
median_step_times = (
    timed_steps.pivot_table(
        index="process_step",
        columns="variation",
        values="step_duration",
        aggfunc="median"
    )
    .reindex(
        index=["start", "step_1", "step_2", "step_3", "confirm"],
        columns=["Test", "Control"],
    )
)

display(median_step_times)

variation,Test,Control
process_step,,
start,14.0,20.0
step_1,27.0,20.0
step_2,61.0,64.0
step_3,58.0,72.0
confirm,93.0,46.0


In [12]:
step_duration_details = (
    timed_steps.groupby(["process_step", "variation"])["step_duration"]
    .describe()
)

display(step_duration_details.round(3))

count     mean      std  min   25%   50%     75%  \
process_step variation                                                       
confirm      Control     1920.0  168.727  410.622  0.0  13.0  46.0  129.25   
             Test        4089.0  243.690  452.196  0.0  42.0  93.0  235.00   
start        Control    35737.0   66.827  286.984  0.0   9.0  20.0   49.00   
             Test       46323.0   61.471  215.202  0.0   7.0  14.0   44.00   
step_1       Control    26043.0   50.468  153.472  0.0   9.0  20.0   47.00   
             Test       35527.0   60.675  143.254  0.0  13.0  27.0   56.00   
step_2       Control    24313.0   92.005  181.146  0.0  33.0  64.0  112.00   
             Test       29573.0   88.849  134.785  0.0  33.0  61.0  103.00   
step_3       Control    20280.0  137.135  212.662  0.0  39.0  72.0  145.00   
             Test       23983.0  129.637  223.699  0.0  31.0  58.0  127.00   

                            max  
process_step variation           
confirm      Control     9396.0  
             Test       14581.0  
start        Control    40235.0  
             Test       24819.0  
step_1       Control    10286.0  
             Test        8797.0  
step_2       Control    21763.0  
             Test       11204.0  
step_3       Control     6164.0  
             Test        6692.0

In [13]:
# how many sessions ended before reaching the step "confirm"

timed_steps[
    (timed_steps["next_step_time"].isna()) &
    (timed_steps["process_step"] != "confirm")
]["visit_id"].nunique()

32508

In [14]:
#how many sessions ended in each step

timed_steps[
    (timed_steps["next_step_time"].isna()) &
    (timed_steps["process_step"] != "confirm")
].groupby("process_step")["visit_id"].nunique()

process_step
start     19092
step_1     6640
step_2     2786
step_3     4000
Name: visit_id, dtype: int64

In [15]:
# Add a column to mention a step when it is the last one 

timed_steps["last_step"] = timed_steps["process_step"].where(
    timed_steps["next_step_time"].isna()
)

# Adding the column to the sessions_data df only when the values are not missing
sessions_data = sessions_data.merge(
    timed_steps[timed_steps["last_step"].notna()][
        ["client_id", "visit_id", "last_step"]
    ],
    on=["client_id", "visit_id"],
    how="left",
    validate="one_to_one"
)
sessions_data.head(10)

,client_id,visit_id,step_count,last_step
0,555,637149525_38041617439_716659,5,confirm
1,647,40369564_40101682850_311847,5,confirm
2,934,7076463_57954418406_971348,4,start
3,1028,557292053_87239438319_391157,9,step_1
4,1104,543158812_46395476577_767725,1,start
5,1104,643221571_99977972121_69283,1,start
6,1186,507052512_11309370126_442139,1,start
7,1186,795373564_99931517312_810896,3,step_2
8,1195,393817425_39015278493_996341,5,confirm
9,1197,71862471_21202285428_848395,7,confirm


In [16]:
# checking for mistakes in the code

timed_steps[timed_steps["client_id"] == 934]

,Unnamed: 0,client_id,visitor_id,visit_id,process_step,date_time,variation,next_step_time,step_duration,last_step
10,56,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:36:30,Test,2017-04-18 02:37:02,32.0,NaN
11,55,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:37:02,Test,2017-04-18 02:38:24,82.0,NaN
12,54,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:38:24,Test,2017-04-18 02:38:52,28.0,NaN
13,53,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:38:52,Test,NaT,NaN,start


In [17]:
#how many last_step in each step

sessions_data["last_step"].value_counts()

last_step
confirm    36927
start      19093
step_1      6640
step_3      4001
step_2      2786
Name: count, dtype: int64

In [18]:
# check the session before the next steps

sessions_data.head()

,client_id,visit_id,step_count,last_step
0,555,637149525_38041617439_716659,5,confirm
1,647,40369564_40101682850_311847,5,confirm
2,934,7076463_57954418406_971348,4,start
3,1028,557292053_87239438319_391157,9,step_1
4,1104,543158812_46395476577_767725,1,start


In [19]:
# ========================================
# 3. Add a column for completed vs. incomplete user journeys
# ========================================

In [20]:
variation_map = clients_data[['client_id', 'variation']].drop_duplicates(subset=['client_id'])
sessions_data = pd.merge(sessions_data, variation_map, on='client_id', how='left')
web_data = pd.merge(web_data, variation_map, on='client_id', how='left')

# 2. check if a (client_id + visit_id) pair reached confirm status
journey_status = (
    web_data.groupby(['client_id', 'visit_id'])['process_step']
    .apply(lambda steps: (steps == 'confirm').any())
    .reset_index(name='completed')
)

# 3. Merge with session_data.csv
sessions_data = pd.merge(sessions_data, journey_status, on=['client_id', 'visit_id'], how='left')

# with string for expandability- if we want to add more category 
sessions_data['journey_status'] = sessions_data['completed'].map({True: 'Completed', False: 'Incomplete'})

sessions_data.head()

,client_id,visit_id,step_count,last_step,variation,completed,journey_status
0,555,637149525_38041617439_716659,5,confirm,Test,True,Completed
1,647,40369564_40101682850_311847,5,confirm,Test,True,Completed
2,934,7076463_57954418406_971348,4,start,Test,False,Incomplete
3,1028,557292053_87239438319_391157,9,step_1,Control,False,Incomplete
4,1104,543158812_46395476577_767725,1,start,Control,False,Incomplete


In [21]:
print(sessions_data['completed'].value_counts())

print("=" * 60)
print('\n')

print(sessions_data['journey_status'].value_counts())

completed
True     37879
False    31568
Name: count, dtype: int64


journey_status
Completed     37879
Incomplete    31568
Name: count, dtype: int64


In [22]:
# ========================================
# 4. Records that don’t follow the right order
# ========================================

In [23]:
# 2. Chronological ordering within a session
web_data['date_time'] = pd.to_datetime(web_data['date_time'])
web_data = web_data.sort_values(by=['client_id','visit_id', 'date_time']).reset_index(drop=True)

# 3. Numbering of steps
step_map = {'start': 0, 'step_1': 1, 'step_2': 2, 'step_3': 3, 'confirm': 4}
web_data['step_num'] = web_data['process_step'].map(step_map)

# 4. The previous step within the same (client_id, visit_id) session
web_data['prev_step_num'] = web_data.groupby(['client_id', 'visit_id'])['step_num'].shift(1)
web_data['step_diff'] = web_data['step_num'] - web_data['prev_step_num']

# 5. Identifying various sequence deviations 
# for the first row if the prev_step_num is NaN = exclude

# A) Any irregular step (i.e., the step forward is not +1)
web_data['is_out_of_order'] = web_data['prev_step_num'].notna() & (web_data['step_diff'] != 1)

# B) Real step backward (e.g step_2 --> step_1, step_diff < 0)
web_data['is_step_back'] = web_data['prev_step_num'].notna() & (web_data['step_diff'] < 0)

# C) Repeat (step1 --> step1, step_diff == 0)
web_data['is_repeat'] = web_data['prev_step_num'].notna() & (web_data['step_diff'] == 0)

# D) Skip a step (step1 --> step3, step_diff > 1)
web_data['is_skip'] = web_data['prev_step_num'].notna() & (web_data['step_diff'] > 1)

# 6. Aggregation to the session level + joining to the df
order_summary = web_data.groupby(['client_id', 'visit_id']).agg(
    out_of_order_count=('is_out_of_order', 'sum'),
    step_backs_count=('is_step_back', 'sum'),
    repeats_count=('is_repeat', 'sum'),
    skips_count=('is_skip', 'sum'),
    starts_at_start=('step_num', lambda x: x.iloc[0] == 0)
).reset_index()

# Was there any sequencing error in the session? T/F
order_summary['follows_right_order'] = (order_summary['out_of_order_count'] == 0) & (order_summary['starts_at_start'])

# 7. Merge with the df
sessions_data = pd.merge(sessions_data, order_summary, on=['client_id', 'visit_id'], how='left')


# check
sessions_data.head()

,client_id,visit_id,step_count,last_step,variation,completed,journey_status,out_of_order_count,step_backs_count,repeats_count,skips_count,starts_at_start,follows_right_order
0,555,637149525_38041617439_716659,5,confirm,Test,True,Completed,0,0,0,0,True,True
1,647,40369564_40101682850_311847,5,confirm,Test,True,Completed,0,0,0,0,True,True
2,934,7076463_57954418406_971348,4,start,Test,False,Incomplete,3,0,3,0,True,False
3,1028,557292053_87239438319_391157,9,step_1,Control,False,Incomplete,4,2,2,0,True,False
4,1104,543158812_46395476577_767725,1,start,Control,False,Incomplete,0,0,0,0,True,True


**out_of_order_count** --> Total out-of-sequence occurrences (sum of the 3 types below):
Includes every instance where the next step was not the standard +1 progression (step_diff != 1). 
step_backs_count + repeats_count + skips_count

**step_backs_count** --> Step back

**repeats_count** --> Repetition

**skips_count** --> Skipped one (or more) steps

In [24]:
print(sessions_data['follows_right_order'].value_counts())

print("=" * 60)
print('\n')
print(sessions_data['starts_at_start'].value_counts())

follows_right_order
True     36605
False    32842
Name: count, dtype: int64


starts_at_start
True     63964
False     5483
Name: count, dtype: int64


starts_at_start == False

Possible reasons:

1. Process continuation in a different session
    a user initiates a process and then closes their browser, later reopen the interface (with a different visit_id), and the system resumes where they left off—meaning and second session begins directly at "step 2" or "step 3"
2. Sessions started before the experimental period (3/15/2017)
    a user initiated the process (the "start" action) before March 15 (activating the logging job) only the subsequent steps were recorded in the database.
3. Timeout, delete cookies, incognito mode
4. Direct link to the stepX (perhaps in an email or from a bookmark in the browser)

Questions // To do:
- check if there is a session with start but not confirm steps with different visit_id but same client_is (earlier timestamps)
- if a user didn't start at start but reached the confirmation stage, does that count as a completed process or not?
- when calculating step_duration or session_duration it is better to handle these cases separately or filter them out --> these sessions completion time would appear shorter than a regular session


In [25]:
# Check how many sessions fall into each category (Test vs Control)
session_counts = (
    sessions_data.pivot_table(
        columns="variation",
        values=[
            "completed",
            "starts_at_start",
            "follows_right_order",
            "out_of_order_count",
            "step_backs_count",
            "repeats_count",
            "skips_count"
        ],
        aggfunc="sum"
    )
    .reindex(
        index=[
            "completed",
            "starts_at_start",
            "follows_right_order",
            "out_of_order_count",
            "step_backs_count",
            "repeats_count",
            "skips_count"
        ],
        columns=["Test", "Control"]
    )
)

# Total sessions row added for reference
session_counts.loc["total_sessions"] = sessions_data.groupby("variation")["visit_id"].count()

print("Number of sessions fall into each category (Test vs Control) ")

display(session_counts)

Number of sessions fall into each category (Test vs Control) 


variation,Test,Control
completed,21791,16088
starts_at_start,33057,30907
follows_right_order,17415,19190
out_of_order_count,36077,23641
step_backs_count,16232,9581
repeats_count,19016,13674
skips_count,829,386
total_sessions,37204,32243


In [26]:
# Check the mean / rate for each metric for users in group Test vs Control
session_means = (
    sessions_data.pivot_table(
        columns="variation",
        values=[
            "completed",
            "starts_at_start",
            "follows_right_order",
            "out_of_order_count",
            "step_backs_count",
            "repeats_count",
            "skips_count"
        ],
        aggfunc="mean"
    )
    .reindex(
        index=[
            "completed",
            "starts_at_start",
            "follows_right_order",
            "out_of_order_count",
            "step_backs_count",
            "repeats_count",
            "skips_count"
        ],
        columns=["Test", "Control"]
    )
)

session_means["Diff (Test - Control)"] = (
    session_means["Test"] - session_means["Control"]
)
print("The mean and rates fall into each category (Test vs Control)")
display(session_means.round(4))

The mean and rates fall into each category (Test vs Control)


variation,Test,Control,Diff (Test - Control)
completed,0.5857,0.4990,0.0868
starts_at_start,0.8885,0.9586,-0.0700
follows_right_order,0.4681,0.5952,-0.1271
out_of_order_count,0.9697,0.7332,0.2365
step_backs_count,0.4363,0.2971,0.1391
repeats_count,0.5111,0.4241,0.0870
skips_count,0.0223,0.0120,0.0103


In [27]:
# Code from yesterday - calculating the total session time

#Finding the start and end time of each session

session_time = web_data.groupby(["client_id", "visit_id"]).agg(start_time=("date_time", "min"),end_time=("date_time", "max")).reset_index()

session_time.head()

,client_id,visit_id,start_time,end_time
0,555,637149525_38041617439_716659,2017-04-15 12:57:56,2017-04-15 13:00:34
1,647,40369564_40101682850_311847,2017-04-12 15:41:28,2017-04-12 15:47:45
2,934,7076463_57954418406_971348,2017-04-18 02:36:30,2017-04-18 02:38:52
3,1028,557292053_87239438319_391157,2017-04-08 18:51:28,2017-04-08 19:00:26
4,1104,543158812_46395476577_767725,2017-06-12 07:49:18,2017-06-12 07:49:18


In [28]:
# Calculating the total time in seconds

session_time["total_time_seconds"] = (session_time["end_time"] - session_time["start_time"]).dt.total_seconds()

session_time.head()

,client_id,visit_id,start_time,end_time,total_time_seconds
0,555,637149525_38041617439_716659,2017-04-15 12:57:56,2017-04-15 13:00:34,158.0
1,647,40369564_40101682850_311847,2017-04-12 15:41:28,2017-04-12 15:47:45,377.0
2,934,7076463_57954418406_971348,2017-04-18 02:36:30,2017-04-18 02:38:52,142.0
3,1028,557292053_87239438319_391157,2017-04-08 18:51:28,2017-04-08 19:00:26,538.0
4,1104,543158812_46395476577_767725,2017-06-12 07:49:18,2017-06-12 07:49:18,0.0


In [29]:
# Checking the median, average, ... of the total time
session_time["total_time_seconds"].describe()

count    69447.000000
mean       299.231500
std        562.283663
min          0.000000
25%         35.000000
50%        164.000000
75%        347.000000
max      41965.000000
Name: total_time_seconds, dtype: float64

In [30]:
# Adding the column to the sessions_data table

sessions_data = sessions_data.merge(session_time[["client_id", "visit_id", "total_time_seconds"]], on=["client_id", "visit_id"], how="left")

sessions_data.head(10)

,client_id,visit_id,step_count,last_step,variation,completed,journey_status,out_of_order_count,step_backs_count,repeats_count,skips_count,starts_at_start,follows_right_order,total_time_seconds
0,555,637149525_38041617439_716659,5,confirm,Test,True,Completed,0,0,0,0,True,True,158.0
1,647,40369564_40101682850_311847,5,confirm,Test,True,Completed,0,0,0,0,True,True,377.0
2,934,7076463_57954418406_971348,4,start,Test,False,Incomplete,3,0,3,0,True,False,142.0
3,1028,557292053_87239438319_391157,9,step_1,Control,False,Incomplete,4,2,2,0,True,False,538.0
4,1104,543158812_46395476577_767725,1,start,Control,False,Incomplete,0,0,0,0,True,True,0.0
5,1104,643221571_99977972121_69283,1,start,Control,False,Incomplete,0,0,0,0,True,True,0.0
6,1186,507052512_11309370126_442139,1,start,Control,False,Incomplete,0,0,0,0,True,True,0.0
7,1186,795373564_99931517312_810896,3,step_2,Control,False,Incomplete,0,0,0,0,True,True,22.0
8,1195,393817425_39015278493_996341,5,confirm,Control,True,Completed,0,0,0,0,True,True,245.0
9,1197,71862471_21202285428_848395,7,confirm,Control,True,Completed,1,1,0,0,True,False,95.0


In [31]:
#Checking the data

print("Sessions with zero seconds:", (sessions_data["total_time_seconds"] == 0).sum())
print("\nMaximum session duration:", sessions_data["total_time_seconds"].max())

Sessions with zero seconds: 11398

Maximum session duration: 41965.0


In [32]:
# ---- NEW CODE!!----
# Order events and calculate time and step changes within each session

web_data = web_data.sort_values(["client_id", "visit_id", "date_time"])
time_between_events = (
    web_data.groupby(["client_id", "visit_id"])["date_time"]
    .diff()
    .dt.total_seconds()
)

step_change = (
    web_data.groupby(["client_id", "visit_id"])["step_num"]
    .diff()
)

In [33]:
# Sum the time spent moving forward by one step in each session
web_data["forward_time_seconds"] = time_between_events.where(
    step_change == 1, 0
)

effective_time = (
    web_data.groupby(["client_id", "visit_id"])["forward_time_seconds"]
    .sum()
    .reset_index(name="effective_time_seconds")
)

display(effective_time.head())

,client_id,visit_id,effective_time_seconds
0,555,637149525_38041617439_716659,158.0
1,647,40369564_40101682850_311847,377.0
2,934,7076463_57954418406_971348,0.0
3,1028,557292053_87239438319_391157,335.0
4,1104,543158812_46395476577_767725,0.0


In [34]:
# Add Effective Time to the master table
sessions_data = sessions_data.merge(
    effective_time,
    on=["client_id", "visit_id"],
    how="left",
    validate="one_to_one"
)

display(sessions_data[
    ["client_id", "visit_id", "total_time_seconds", "effective_time_seconds"]
].head(10))

,client_id,visit_id,total_time_seconds,effective_time_seconds
0,555,637149525_38041617439_716659,158.0,158.0
1,647,40369564_40101682850_311847,377.0,377.0
2,934,7076463_57954418406_971348,142.0,0.0
3,1028,557292053_87239438319_391157,538.0,335.0
4,1104,543158812_46395476577_767725,0.0,0.0
5,1104,643221571_99977972121_69283,0.0,0.0
6,1186,507052512_11309370126_442139,0.0,0.0
7,1186,795373564_99931517312_810896,22.0,22.0
8,1195,393817425_39015278493_996341,245.0,245.0
9,1197,71862471_21202285428_848395,95.0,84.0


In [35]:
## DAY 2 CODE ##

# Check only start events
starts = web_data[web_data["process_step"] == "start"]

starts.head()

,Unnamed: 0,client_id,visitor_id,visit_id,process_step,date_time,variation_x,variation_y,step_num,prev_step_num,step_diff,is_out_of_order,is_step_back,is_repeat,is_skip,forward_time_seconds
0,16,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,Test,0,NaN,NaN,False,False,False,False,0.0
5,21,647,66758770_53988066587,40369564_40101682850_311847,start,2017-04-12 15:41:28,Test,Test,0,NaN,NaN,False,False,False,False,0.0
10,56,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:36:30,Test,Test,0,NaN,NaN,False,False,False,False,0.0
11,55,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:37:02,Test,Test,0,0.0,0.0,True,False,True,False,0.0
12,54,934,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:38:24,Test,Test,0,0.0,0.0,True,False,True,False,0.0


In [36]:
#Checking how many counts exist for a client in each visit 

starts_count = (starts.groupby(["client_id", "visit_id"]).size().reset_index(name="starts_count"))

starts_count.head(10)

,client_id,visit_id,starts_count
0,555,637149525_38041617439_716659,1
1,647,40369564_40101682850_311847,1
2,934,7076463_57954418406_971348,4
3,1028,557292053_87239438319_391157,1
4,1104,543158812_46395476577_767725,1
5,1104,643221571_99977972121_69283,1
6,1186,507052512_11309370126_442139,1
7,1186,795373564_99931517312_810896,1
8,1195,393817425_39015278493_996341,1
9,1197,71862471_21202285428_848395,1


In [37]:
# Adding the new column to the sessions table

sessions_data = sessions_data.merge(starts_count, on=["client_id", "visit_id"], how="left", validate="one_to_one")

sessions_data["starts_count"] = (sessions_data["starts_count"].fillna(0).astype(int))

sessions_data.head(10)

,client_id,visit_id,step_count,last_step,variation,completed,journey_status,out_of_order_count,step_backs_count,repeats_count,skips_count,starts_at_start,follows_right_order,total_time_seconds,effective_time_seconds,starts_count
0,555,637149525_38041617439_716659,5,confirm,Test,True,Completed,0,0,0,0,True,True,158.0,158.0,1
1,647,40369564_40101682850_311847,5,confirm,Test,True,Completed,0,0,0,0,True,True,377.0,377.0,1
2,934,7076463_57954418406_971348,4,start,Test,False,Incomplete,3,0,3,0,True,False,142.0,0.0,4
3,1028,557292053_87239438319_391157,9,step_1,Control,False,Incomplete,4,2,2,0,True,False,538.0,335.0,1
4,1104,543158812_46395476577_767725,1,start,Control,False,Incomplete,0,0,0,0,True,True,0.0,0.0,1
5,1104,643221571_99977972121_69283,1,start,Control,False,Incomplete,0,0,0,0,True,True,0.0,0.0,1
6,1186,507052512_11309370126_442139,1,start,Control,False,Incomplete,0,0,0,0,True,True,0.0,0.0,1
7,1186,795373564_99931517312_810896,3,step_2,Control,False,Incomplete,0,0,0,0,True,True,22.0,22.0,1
8,1195,393817425_39015278493_996341,5,confirm,Control,True,Completed,0,0,0,0,True,True,245.0,245.0,1
9,1197,71862471_21202285428_848395,7,confirm,Control,True,Completed,1,1,0,0,True,False,95.0,84.0,1


In [38]:
#Checking the data
print(sessions_data["starts_count"].value_counts())
print("\nMáx:",sessions_data["starts_count"].max())
print("\nMin:",sessions_data["starts_count"].min())

starts_count
1     43576
2     12174
0      5266
3      4538
4      2058
5       897
6       422
7       229
8       106
9        78
10       37
11       25
12       14
13       10
14        7
15        3
16        2
19        2
17        2
23        1
Name: count, dtype: int64

Máx: 23

Min: 0


In [39]:
# Adding new column showing entries with multiple starts

sessions_data["multiple_starts"] = (sessions_data["starts_count"] > 1)

sessions_data["multiple_starts"].value_counts()

multiple_starts
False    48842
True     20605
Name: count, dtype: int64

In [40]:
# Doing the same for confirm events

confirms = web_data[web_data["process_step"] == "confirm"]

confirms_count = (confirms.groupby(["client_id", "visit_id"]).size().reset_index(name="confirms_count"))

sessions_data = sessions_data.merge(confirms_count, on=["client_id", "visit_id"], how="left", validate="one_to_one")

sessions_data["confirms_count"] = (sessions_data["confirms_count"].fillna(0).astype(int))

In [41]:
#Checking the data
print(sessions_data["confirms_count"].value_counts())
print("\nMáx:",sessions_data["confirms_count"].max())
print("\nMin:",sessions_data["confirms_count"].min())

sessions_data["multiple_confirms"] = (sessions_data["confirms_count"] > 1)

sessions_data["multiple_confirms"].value_counts()

confirms_count
1     34333
0     31568
2      2606
3       632
4       192
5        50
8        22
6        20
7        19
9         3
10        1
11        1
Name: count, dtype: int64

Máx: 11

Min: 0


multiple_confirms
False    65901
True      3546
Name: count, dtype: int64

In [42]:
sessions_data.head(10)

,client_id,visit_id,step_count,last_step,variation,completed,journey_status,out_of_order_count,step_backs_count,repeats_count,skips_count,starts_at_start,follows_right_order,total_time_seconds,effective_time_seconds,starts_count,multiple_starts,confirms_count,multiple_confirms
0,555,637149525_38041617439_716659,5,confirm,Test,True,Completed,0,0,0,0,True,True,158.0,158.0,1,False,1,False
1,647,40369564_40101682850_311847,5,confirm,Test,True,Completed,0,0,0,0,True,True,377.0,377.0,1,False,1,False
2,934,7076463_57954418406_971348,4,start,Test,False,Incomplete,3,0,3,0,True,False,142.0,0.0,4,True,0,False
3,1028,557292053_87239438319_391157,9,step_1,Control,False,Incomplete,4,2,2,0,True,False,538.0,335.0,1,False,0,False
4,1104,543158812_46395476577_767725,1,start,Control,False,Incomplete,0,0,0,0,True,True,0.0,0.0,1,False,0,False
5,1104,643221571_99977972121_69283,1,start,Control,False,Incomplete,0,0,0,0,True,True,0.0,0.0,1,False,0,False
6,1186,507052512_11309370126_442139,1,start,Control,False,Incomplete,0,0,0,0,True,True,0.0,0.0,1,False,0,False
7,1186,795373564_99931517312_810896,3,step_2,Control,False,Incomplete,0,0,0,0,True,True,22.0,22.0,1,False,0,False
8,1195,393817425_39015278493_996341,5,confirm,Control,True,Completed,0,0,0,0,True,True,245.0,245.0,1,False,1,False
9,1197,71862471_21202285428_848395,7,confirm,Control,True,Completed,1,1,0,0,True,False,95.0,84.0,1,False,1,False


In [43]:
# checking sessions with multiple starts but no other step

starts_no_progress = sessions_data[
    (sessions_data["starts_count"] > 1) &
    (sessions_data["starts_count"] == sessions_data["step_count"])
]

display(starts_no_progress.head())
starts_no_progress.shape

,client_id,visit_id,step_count,last_step,variation,completed,journey_status,out_of_order_count,step_backs_count,repeats_count,skips_count,starts_at_start,follows_right_order,total_time_seconds,effective_time_seconds,starts_count,multiple_starts,confirms_count,multiple_confirms
2,934,7076463_57954418406_971348,4,start,Test,False,Incomplete,3,0,3,0,True,False,142.0,0.0,4,True,0,False
30,2720,565429886_73206140156_730955,3,start,Test,False,Incomplete,2,0,2,0,True,False,173.0,0.0,3,True,0,False
67,8610,243751268_9020460563_923194,2,start,Test,False,Incomplete,1,0,1,0,True,False,18.0,0.0,2,True,0,False
77,10344,595675198_11906913204_686811,5,start,Control,False,Incomplete,4,0,4,0,True,False,3340.0,0.0,5,True,0,False
98,15753,395225539_72187591129_857978,3,start,Test,False,Incomplete,2,0,2,0,True,False,2478.0,0.0,3,True,0,False


(4528, 19)

In [44]:
sessions_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 69447 entries, 0 to 69446
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   client_id               69447 non-null  int64  
 1   visit_id                69447 non-null  str    
 2   step_count              69447 non-null  int64  
 3   last_step               69447 non-null  str    
 4   variation               69447 non-null  str    
 5   completed               69447 non-null  bool   
 6   journey_status          69447 non-null  str    
 7   out_of_order_count      69447 non-null  int64  
 8   step_backs_count        69447 non-null  int64  
 9   repeats_count           69447 non-null  int64  
 10  skips_count             69447 non-null  int64  
 11  starts_at_start         69447 non-null  bool   
 12  follows_right_order     69447 non-null  bool   
 13  total_time_seconds      69447 non-null  float64
 14  effective_time_seconds  69447 non-null  float64
 

In [46]:
# Find the first recorded start in each session.
session_starts = (
    web_data.loc[web_data["process_step"] == "start"]
    .groupby(["client_id", "visit_id"])["date_time"]
    .min()
    .reset_index(name="first_start_time")
)

display(session_starts.head())

,client_id,visit_id,first_start_time
0,555,637149525_38041617439_716659,2017-04-15 12:57:56
1,647,40369564_40101682850_311847,2017-04-12 15:41:28
2,934,7076463_57954418406_971348,2017-04-18 02:36:30
3,1028,557292053_87239438319_391157,2017-04-08 18:51:28
4,1104,543158812_46395476577_767725,2017-06-12 07:49:18


In [47]:
# Find the first confirmation after the session's first start
session_confirms = web_data.loc[
    web_data["process_step"] == "confirm",
    ["client_id", "visit_id", "date_time"],
].merge(
    session_starts,
    on=["client_id", "visit_id"],
    how="inner",
    validate="many_to_one",
)

session_confirms = (
    session_confirms.loc[
        session_confirms["date_time"] > session_confirms["first_start_time"]
    ]
    .groupby(["client_id", "visit_id"])["date_time"]
    .min()
    .reset_index(name="first_confirm_time")
)

display(session_confirms.head())

,client_id,visit_id,first_confirm_time
0,555,637149525_38041617439_716659,2017-04-15 13:00:34
1,647,40369564_40101682850_311847,2017-04-12 15:47:45
2,1195,393817425_39015278493_996341,2017-04-05 20:19:31
3,1197,71862471_21202285428_848395,2017-04-01 17:17:59
4,1336,583743392_96265099036_939815,2017-05-08 06:08:43


In [48]:
# Calculate seconds from the first start to the first confirmation
completion_times = session_starts.merge(
    session_confirms,
    on=["client_id", "visit_id"],
    how="left",
    validate="one_to_one",
)

completion_times["completion_time_seconds"] = (
    completion_times["first_confirm_time"] - completion_times["first_start_time"]
).dt.total_seconds()

display(completion_times.head())

,client_id,visit_id,first_start_time,first_confirm_time,completion_time_seconds
0,555,637149525_38041617439_716659,2017-04-15 12:57:56,2017-04-15 13:00:34,158.0
1,647,40369564_40101682850_311847,2017-04-12 15:41:28,2017-04-12 15:47:45,377.0
2,934,7076463_57954418406_971348,2017-04-18 02:36:30,NaT,NaN
3,1028,557292053_87239438319_391157,2017-04-08 18:51:28,NaT,NaN
4,1104,543158812_46395476577_767725,2017-06-12 07:49:18,NaT,NaN


In [49]:
# Add completion timestamps and duration to the master table.
sessions_data = sessions_data.merge(
    completion_times,
    on=["client_id", "visit_id"],
    how="left",
    validate="one_to_one",
)

display(sessions_data[
    ["client_id", "visit_id", "first_start_time",
     "first_confirm_time", "completion_time_seconds"]
].head(10))

,client_id,visit_id,first_start_time,first_confirm_time,completion_time_seconds
0,555,637149525_38041617439_716659,2017-04-15 12:57:56,2017-04-15 13:00:34,158.0
1,647,40369564_40101682850_311847,2017-04-12 15:41:28,2017-04-12 15:47:45,377.0
2,934,7076463_57954418406_971348,2017-04-18 02:36:30,NaT,NaN
3,1028,557292053_87239438319_391157,2017-04-08 18:51:28,NaT,NaN
4,1104,543158812_46395476577_767725,2017-06-12 07:49:18,NaT,NaN
5,1104,643221571_99977972121_69283,2017-06-20 22:31:33,NaT,NaN
6,1186,507052512_11309370126_442139,2017-04-08 15:59:16,NaT,NaN
7,1186,795373564_99931517312_810896,2017-04-08 18:05:02,NaT,NaN
8,1195,393817425_39015278493_996341,2017-04-05 20:15:26,2017-04-05 20:19:31,245.0
9,1197,71862471_21202285428_848395,2017-04-01 17:16:24,2017-04-01 17:17:59,95.0


In [50]:
# save to csv
sessions_data.to_csv('../data/clean/sessions_data_master.csv', index=False)

In [51]:
# Pull variation from web_data
if "variation" not in web_data.columns:
    var_map = clients_data[["client_id", "variation"]].drop_duplicates(subset=["client_id"])
    web_data = web_data.merge(var_map, on="client_id", how="left")

# Only Test and Control variation
web_ab = web_data[web_data["variation"].isin(["Control", "Test"])].copy()
web_ab["date_time"] = pd.to_datetime(web_ab["date_time"])

# Chronological ordering within a session
web_ab = web_ab.sort_values(["client_id", "visit_id", "date_time"]).reset_index(drop=True)

step_order = {"start": 0, "step_1": 1, "step_2": 2, "step_3": 3, "confirm": 4}
web_ab["step_num"] = web_ab["process_step"].map(step_order)

### Identifiers and group assignments: 
Check whether a client is ever assigned to both Test and Control, and whether the same visit ID appears under different clients. Investigate any cases you find.

In [52]:
# ==============================================================================
# CHECK OF IDENTIFIERS & GROUP ASSIGNMENTS
#    - Does a client_id appear in both the Test and Control groups?
#    - Does the same visit_id appear under different client_ids?
# ==============================================================================
print("=" * 80)
print("IDENTIFIERS & GROUP ASSIGNMENTS AUDIT")
print("=" * 80)

# A) Client ID -> Variation clarity
client_var_counts = web_ab.groupby("client_id")["variation"].nunique()
multi_var_clients = client_var_counts[client_var_counts > 1]
print(f"Clients assigned to BOTH Test and Control: {len(multi_var_clients)}")

if len(multi_var_clients) > 0:
    display(web_ab[web_ab["client_id"].isin(multi_var_clients.index)].head(20))

# B) Visit ID -> Client ID clarity
visit_client_counts = web_ab.groupby("visit_id")["client_id"].nunique()
shared_visits = visit_client_counts[visit_client_counts > 1]
print(f"Visit IDs appearing under DIFFERENT client_ids: {len(shared_visits)}")

if len(shared_visits) > 0:
    print("\n--- Investigation of Shared Visit IDs (Sample) ---")
    shared_df = web_ab[web_ab["visit_id"].isin(shared_visits.index)].sort_values(["visit_id", "date_time"])
    display(shared_df[["visit_id", "client_id", "visitor_id", "variation", "process_step", "date_time"]].head(20))
    
    # Check if they come from the same visitor_id (shared device)
    same_visitor = (shared_df.groupby("visit_id")["visitor_id"].nunique() == 1).sum()
    print(f"Out of {len(shared_visits)} shared visit_ids, {same_visitor} share the exact same visitor_id ( = shared device/browser).")

IDENTIFIERS & GROUP ASSIGNMENTS AUDIT
Clients assigned to BOTH Test and Control: 0
Visit IDs appearing under DIFFERENT client_ids: 242

--- Investigation of Shared Visit IDs (Sample) ---


,visit_id,client_id,visitor_id,variation,process_step,date_time
52903,102039130_96083762708_562998,1688588,494497404_19580991399,Test,start,2017-05-29 14:33:38
52904,102039130_96083762708_562998,1688588,494497404_19580991399,Test,step_1,2017-05-29 14:33:44
52905,102039130_96083762708_562998,1688588,494497404_19580991399,Test,step_1,2017-05-29 14:35:08
52906,102039130_96083762708_562998,1688588,494497404_19580991399,Test,step_2,2017-05-29 14:35:20
52907,102039130_96083762708_562998,1688588,494497404_19580991399,Test,step_3,2017-05-29 14:37:16
52908,102039130_96083762708_562998,1688588,494497404_19580991399,Test,confirm,2017-05-29 14:38:10
308747,102039130_96083762708_562998,9719474,494497404_19580991399,Control,start,2017-05-29 14:44:08
308748,102039130_96083762708_562998,9719474,494497404_19580991399,Control,step_1,2017-05-29 14:44:12
308749,102039130_96083762708_562998,9719474,494497404_19580991399,Control,step_2,2017-05-29 14:44:23
308750,102039130_96083762708_562998,9719474,494497404_19580991399,Control,step_3,2017-05-29 14:45:21


Out of 242 shared visit_ids, 242 share the exact same visitor_id ( = shared device/browser).


**Identifiers & Group Assignments Summary:**

* **Zero Group Cross-Contamination:** **0 clients** (`client_id`) were assigned to both the **Test** and **Control** groups, confirming that client-level randomization remained completely clean.


* **Shared Device Sessions Properly Isolated:** **242 `visit_id` sessions** appeared under multiple `client_id` values. In **100% of these cases (242 out of 242)**, the events shared the exact same `visitor_id`—proving they represent different clients (e.g., family members) logging in sequentially on a shared device/browser before the session expired, rather than server-side ID collisions.


* **Methodology Validation:** Keying sessions on the composite identifier `['client_id', 'visit_id']` automatically separated these shared-device visits into distinct, uncontaminated client journeys.

### Funnel conversion: 
Check what percentage of sessions continued from each step to the next, in time order. The table we have only counts how many sessions recorded each step independently.

We need to follow each session from start and check how far it progressed through the required steps in order.

For each session, check:
- Did it record start?
- After that, did it record step_1?
- After that, did it record step_2?
- After that, did it record step_3?
- After that, did it record confirm?
- Repeats and backward moves are allowed. A session like start → step_1 → start → step_1 → step_2 reached step_2 in order. Each session counts once at each stage.

In [53]:
# ==============================================================================
# Funnel conversion
# ==============================================================================
print("\n" + "=" * 80)
print("FUNNEL CONVERSION (CHRONOLOGICAL STEP PROGRESSION)")
print("=" * 80)

# A) Strict Chronological Funnel:
# A session has reached level `step_k` if it began with a start event (0) and, in chronological order,
# passed through all preceding steps (0 -> 1 -> ... -> k) before to the step_k.

def check_chronological_funnel(steps):
    reached_0 = False
    reached_1 = False
    reached_2 = False
    reached_3 = False
    reached_4 = False
    
    for s in steps:
        if s == 0:
            reached_0 = True
        elif s == 1 and reached_0:
            reached_1 = True
        elif s == 2 and reached_1:
            reached_2 = True
        elif s == 3 and reached_2:
            reached_3 = True
        elif s == 4 and reached_3:
            reached_4 = True
            
    return pd.Series({
        "0_start": reached_0,
        "1_step_1": reached_1,
        "2_step_2": reached_2,
        "3_step_3": reached_3,
        "4_confirm": reached_4
    })

funnel_flags = (
    web_ab.groupby(["client_id", "visit_id", "variation"])["step_num"]
    .apply(check_chronological_funnel)
    .unstack()
    .reset_index()
)

# Chronological deviation compare to the total sessions (%)
strict_funnel_pct = (
    funnel_flags.groupby("variation")[["0_start", "1_step_1", "2_step_2", "3_step_3", "4_confirm"]]
    .mean()
    .T[["Control", "Test"]] * 100
)
strict_funnel_pct["Diff (Test - Control) pp"] = strict_funnel_pct["Test"] - strict_funnel_pct["Control"]

print("--- A) Strict Chronological Funnel (% of ALL sessions reaching each step in order) ---")
display(strict_funnel_pct.round(2))

# B) Step-to-Step progression rate:
# Of those who reached a given step in the sequence, what percentage proceeded to the next one?
step_to_step = pd.DataFrame(index=[
    "start -> step_1",
    "step_1 -> step_2",
    "step_2 -> step_3",
    "step_3 -> confirm"
])

for grp in ["Control", "Test"]:
    sub = funnel_flags[funnel_flags["variation"] == grp]
    step_to_step[grp] = [
        (sub["1_step_1"].sum() / sub["0_start"].sum()) * 100,
        (sub["2_step_2"].sum() / sub["1_step_1"].sum()) * 100,
        (sub["3_step_3"].sum() / sub["2_step_2"].sum()) * 100,
        (sub["4_confirm"].sum() / sub["3_step_3"].sum()) * 100,
    ]

step_to_step["Diff (Test - Control) pp"] = step_to_step["Test"] - step_to_step["Control"]

print("\n--- B) Step-to-Step Conversion Rate (% continuing from previous step in time order) ---")
display(step_to_step.round(2))


FUNNEL CONVERSION (CHRONOLOGICAL STEP PROGRESSION)
--- A) Strict Chronological Funnel (% of ALL sessions reaching each step in order) ---


variation,Control,Test,Diff (Test - Control) pp
0_start,96.03,89.29,-6.74
1_step_1,72.29,75.42,3.12
2_step_2,61.38,64.86,3.48
3_step_3,55.13,58.07,2.95
4_confirm,45.98,48.10,2.12



--- B) Step-to-Step Conversion Rate (% continuing from previous step in time order) ---


,Control,Test,Diff (Test - Control) pp
start -> step_1,75.28,84.46,9.18
step_1 -> step_2,84.91,86.00,1.09
step_2 -> step_3,89.80,89.54,-0.27
step_3 -> confirm,83.41,82.83,-0.58


**Chronological Funnel Conversion Summary:**

* **Overall Reach:** In strict chronological order (`start` $\rightarrow$ `step_1` $\rightarrow$ `step_2` $\rightarrow$ `step_3` $\rightarrow$ `confirm`), the **Test** group achieved higher reach at every step from `step_1` onward, ending with **48.10%** reaching `confirm` compared to **45.98%** in **Control** (**+2.12 pp**), despite fewer Test sessions starting at `start` (**89.29%** vs. **96.03%**, reflecting higher mid-funnel returning visits).
  
* **Step-to-Step Continuation:** The redesigned UI gained virtually its entire advantage at the very first transition (`start` $\rightarrow$ `step_1`), where continuation jumped by **+9.18 percentage points** (**84.46%** in Test vs. **75.28%** in Control). Subsequent step-to-step continuation rates remained nearly identical between groups (`+1.09 pp`, `-0.27 pp`, and `-0.58 pp`).

* **Business Takeaway:** The new UI and in-context prompts significantly lowered the initial entry barrier and helped clients get started, though late-stage progression (`step_2` through `confirm`) remained unchanged.